### Text Splitter

In [2]:
import pdfplumber
from common.neo4j_client import Neo4jCustomClient
from common.llm_client import LLMClient
from common.embedding_client import  EmbeddingClient

llm = LLMClient()
neo4j_client = Neo4jCustomClient()
embedding = EmbeddingClient()

In [3]:

print(f"LLM {llm.get_model_name()}")
print(f"Neo4j {neo4j_client.verify_connectivity()}")
print(f"Embedding {embedding.get_model_name()}")

LLM google/gemma-4-e4b
Neo4j True
Embedding text-embedding-bge-m3


### Step-Back Prompting

In [4]:
stepback_system_message = """
당신은 세계 상식에 정통한 전문가입니다. 당신의 임무는 질문을 더 일반적인 형태의 질문으로 재구성하여 답변하기 쉽게 만드는 것입니다.

아래는 몇 가지 예시입니다.
"input" : "Thread보다 Process 생성이 느린 이유는?"
"output" : "운영체제에서 Process와 Thread는 어떻게 관리되는가?"
"input" : "GPT가 Hallucination을 하는 이유는?"
"output" : "LLM은 답변을 어떻게 생성하는가?"
"""

In [5]:
def generate_stepback(question: str):
    user_message = f"""{question}"""
    step_back_question = llm.chat(
        messages=[
            {"role": "system", "content": stepback_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    return step_back_question

In [6]:
question = "python의 Ray 라이브러리는 왜 쓰나요?"
step_back_question = generate_stepback(question)
print(f"step back results:\n{step_back_question}")

step back results:
**"파이썬 코드를 대규모 병렬 처리 또는 분산 컴퓨팅 환경에서 실행하려면 어떻게 해야 하는가?"**

*(또는 더 간결하게: "대규모 컴퓨팅 작업을 효율적으로 확장하는 방법은 무엇인가?")*


### PDF Loader

In [8]:
import re
from pathlib import Path

FILE_PATH = Path.cwd().parent/"data"/"RE-185. 인공지능 기술·산업 생태계 육성방안 연구.pdf"


### 목차, 표지 제거

In [9]:
text = ""

# with pdfplumber.open(FILE_PATH) as pdf:
#     for page in pdf.pages:
#         text += page.extract_text()

with pdfplumber.open(FILE_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages):
        # 0~6페이지 제거
        if page_num <= 8:
            continue

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

In [11]:
print(text)

제1장 서 론
제1절 연구 배경 및 필요성
인공지능(AI) 기술은 최근 비약적으로 발전하며 전통적인 산업 구조의 근본적인 변화와
새로운 경제 생태계의 형성을 이끌고 있다. 과거 AI 기술이 인간 수준으로 수행가능한
기능은 이미지 분류나 기본적 독해 정도에 국한되어 있었고, 기능의 수행능력이 미흡
하여 상업적 목적으로 활용하는 데는 한계가 있었다. 이러한 이유로 AI 기술은 두 차례
빙하기(AI winter)를 맞이했던 것으로 알려져 있다. 그러나 AI 기술이 고도화됨에 따라
수행가능한 기능이 다양해지고 동시에 기능별 수행능력이 급격하게 향상되면서(Maslej
et al., 2024), 실제 산업 현장에서의 활용 가능성에 대한 관심 및 기대 또한 높아져 갔다.
이는 AI에 대한 글로벌 벤처캐피털(VC) 투자가 최근 10년 사이 대폭 확대된 것을 통해
알 수 있다. Dealroom(2025)에 따르면, 2024년 AI에 대한 글로벌 VC 투자 규모는 약 1천
1백억 달러 규모로 확인되었으며, 이는 2014년 약 80억 달러 규모에서 약 13.8배 늘어난
수준이다. 전체 글로벌 VC 투자에서 AI 스타트업에 대한 투자가 차지하는 비중은 2014년
약 7% 수준이었으나, 2022년까지 완만하게 증가하다가 2022년부터 급증하면서 2024년
33%를 기록한 것으로 나타났다. 요컨대, 고도화된 AI 기술의 상업적 활용가능성에 대한
기대가 최근에서야 높아졌다고 볼 수 있다는 것이다.
* 자료 : Maslej et al.(2024)
[그림 1-1] AI의 기능별 인간대비 수준 변화 추이
- 1 -
< 전체 대비 AI 스타트업에 대한
< 글로벌 VC AI 투자 규모 >
글로벌 VC 투자 비중 >
(단위: 10억 달러) (단위: %)
* 자료 : Dealroom(2025)
[그림 1-2] 글로벌 VC AI 투자 추이(2014~2024년)
현재 AI는 제조·의료·금융·환경·모빌리티·농업 등 다양한 산업 현장에 침투하여
활용되고 있으며, 기술이 고도화됨에 따라 점차 활용범위가 확대

In [12]:
title_pattern = re.compile(
    r"(?m)^("
    r"제\d+장\s+.+"
    r"|제\d+절\s+.+"
    r")$"
)

titles = title_pattern.findall(text)

for i, title in enumerate(titles):
    print(i, repr(title))

0 '제1장 서 론'
1 '제1절 연구 배경 및 필요성'
2 '제2절 연구 목적 및 내용'
3 '제2장 주요국 정책 추진 현황 분석'
4 '제1절 국가별 현황'
5 '제2절 소결 및 시사점'
6 '제3장 글로벌 AI 연구 현황 분석'
7 '제1절 데이터 수집 방법론 개발 필요성'
8 '제2절 데이터 수집 방법론'
9 '제3절 글로벌 AI 분야 논문 현황 및 시사점'
10 '제4장 국내·외 AI 기업 분석 및 사례 연구'
11 '제1절 개요'
12 '제2절 AI 기업의 경제적 성과 영향요인에 관한 실증분석'
13 '제3절 국내·외 AI 기업 우수사례 분석'
14 '제5장 글로벌 AI 시장 동향 및 국내 AI 수요 현황 분석'
15 '제1절 개요 및 AI 시장 동향'
16 '제2절 국내 기업의 AI 수요 현황 및 인식 조사'
17 '제3절 소결 및 시사점'
18 '제6장 결론'
19 '제1절 연구결과 종합'
20 '제2절 영역별 정책제언'


In [13]:
import re

# def clean_pdf_text(text: str) -> str:
#     text = re.sub(r"[·.]{3,}", " ", text)
#     text = re.sub(r"[ \t]+", " ", text)
#
#     return text.strip()

def split_text_by_titles(text: str) -> list[str]:

    title_pattern = re.compile(
       r"(?m)^("
        r"제\d+장\s+.+"
        r"|제\d+절\s+.+"
        r")$")
    parts = re.split(title_pattern, text)

    sections = []

    # 제목 이전 내용
    if parts[0].strip():
        sections.append(parts[0].strip())

    # 제목 + 본문 결합
    for i in range(1, len(parts), 2):
        title = parts[i].strip()

        content = ""
        if i + 1 < len(parts):
            content = parts[i + 1].strip()

        sections.append(f"{title}\n{content}")

    return sections

In [14]:

sections = split_text_by_titles(text)
print(f"Number of sections: {len(sections)}")
#print(sections[7])
print(len(sections))

Number of sections: 21
21


In [15]:
for i, s in enumerate(sections):
    print("=" * 50)
    print(f"{i}, length: {len(s)}, token size {embedding.num_tokens(s)}")
    print(s[:200])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17537 > 8192). Running this sequence through the model will result in indexing errors


0, length: 8, token size 8
제1장 서 론

1, length: 3164, token size 1484
제1절 연구 배경 및 필요성
인공지능(AI) 기술은 최근 비약적으로 발전하며 전통적인 산업 구조의 근본적인 변화와
새로운 경제 생태계의 형성을 이끌고 있다. 과거 AI 기술이 인간 수준으로 수행가능한
기능은 이미지 분류나 기본적 독해 정도에 국한되어 있었고, 기능의 수행능력이 미흡
하여 상업적 목적으로 활용하는 데는 한계가 있었다. 이러한 이유로 AI 기술
2, length: 881, token size 447
제2절 연구 목적 및 내용
본 연구에서는 우리나라 AI 기술·산업 생태계 육성을 위한 정책 수립에 참고할 수
있는 시사점과 제언을 도출하기 위해 국내·외 정책, 기술 및 R&D 과제, 기업, 수요(시장)
등을 분석한다. 구체적인 연구 내용과 구성은 다음과 같다.
첫째, 미국·중국·EU·싱가포르 등 주요국의 AI 정책 추진 현황을 분석한다. 이를
통해 우리나
3, length: 20, token size 12
제2장 주요국 정책 추진 현황 분석

4, length: 37180, token size 17537
제1절 국가별 현황
1. 미국
AI 기술과 글로벌 산업 생태계를 선도하고 있는 미국의 AI 정책은 오바마 2기 행정부
(2013년 ~ 2017년)를 기점으로 본격적으로 전개되었다고 볼 수 있다. 오바마 행정부에
서는 다음과 같은 AI 정책 관련 3부작 보고서를 발표하였다.
먼저, 2016년 10월에 발표한 「국가 AI R&D 전략(The National A
5, length: 4898, token size 2438
제2절 소결 및 시사점
본 장에서는 미국, 중국, EU, 싱가포르 등 주요국의 AI 정책 동향과 특징을 살펴보았다.
즉, 거시적 관점에서 이들 국가가 어떠한 방향성을 가지고 AI 생태계를 육성하고 있고,
어떠한 방식으로 국가 경쟁력 강화를 도모하고 있는지 분석하고자 하였다.
종합하면, 앞서 살펴본 주요국들은 공통적으로 

In [16]:
for i, section in enumerate(sections):
    if len(section) == 37180:
        print(f"Index: {i}")
        print(section[:1000])
        break

Index: 4
제1절 국가별 현황
1. 미국
AI 기술과 글로벌 산업 생태계를 선도하고 있는 미국의 AI 정책은 오바마 2기 행정부
(2013년 ~ 2017년)를 기점으로 본격적으로 전개되었다고 볼 수 있다. 오바마 행정부에
서는 다음과 같은 AI 정책 관련 3부작 보고서를 발표하였다.
먼저, 2016년 10월에 발표한 「국가 AI R&D 전략(The National Artificial Intelligence
R&D Strategic Plan)」에서는 민간과 정부의 ‘역할분담’을 강조하는 동시에, 시장실패
(market failure)가 발생하지만 사회적 영향이 지대한 분야인 기초 연구와 공공 영역(예:
안보, 공중위생 등)에 대해서는 정부 차원의 대규모·장기적 투자를 확대하는 방침을
권고하였다. 동 보고서에는 연방 정부 자금에 의한 AI R&D에 관하여 7가지 전략적 우선
사항과 2가지 권고 사항도 포함되었다(NSTC, 2016a).
둘째, 마찬가지로 2016년 10월에 발표한 「AI의 미래에 대비(Preparing for the Future
of Artificial Intelligence)」에서는 AI 기술 발전과 낙관적 미래 전망을 제시하는 한편,
AI 활용 기반 조성을 위해 AI 윤리, 보안, 교육 등 다방면에 걸친 정책 수립의 필요성을
제기하였다. 특히, AI 인재의 양적 규모 확대, 질적 수준 제고, 다양성 강화 등을 위한
연방 정부와 교육기관의 역할을 구체적으로 제시하였다(NSTC, 2016).
셋째, 2016년 12월 발표한 「AI, 자동화, 그리고 경제(Artificial Intelligence, Automation,
and the Economy)」에서는 AI에 의한 자동화가 거시경제 및 노동시장(고용)에 미치는
영향을 분석하고, 그 영향에 대응하기 위한 정책적 대응방안을 제시하였다. 구체적으로,
미래 직업을 위한 전문ž직업ž평생교육 지원, 사회안전망 확충, AI의 경제적 파급효과 강화를
위한 투자 확대 등을 제안하였다(EOP, 2016b).

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

PARENT_CHUNK_SIZE = 2000
PARENT_OVERLAP = 40
CHILD_CHUNK_SIZE = 500
CHILD_OVERLAP = 20


parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=PARENT_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

parent_chunks = []

for section in sections:
    parent_chunks.extend(
        parent_splitter.split_text(section)
    )

In [18]:
pdf_id = Path(FILE_PATH).stem
print(pdf_id)

RE-185. 인공지능 기술·산업 생태계 육성방안 연구


In [20]:
query = "10-import-parent-child-embeddings"

for i, chunk in enumerate(parent_chunks):
    child_chunks = child_splitter.split_text(chunk)
    embeddings = embedding.embed_documents(child_chunks)
    # Add to neo4j
    neo4j_client.execute_query(
        query,
        id=str(i),
        pdf_id=pdf_id,
        parent=chunk,
        children=child_chunks,
        embeddings=embeddings,
    )
